# 🍅 Tomato Pipeline Demo

**YOLO Detection → Qwen3.5 Ripeness → Qwen3.5 Harvest Selection**

| Step | Model | Task |
|---|---|---|
| 1 | YOLO v2.6s (2-class) | 이미지에서 토마토 bbox 탐지 → 단일 `tomato` 클래스로 통합 |
| 2 | Qwen3.5-0.8B SFT (Ripeness) | 각 bbox crop → ripe / unripe 분류 |
| 3 | Qwen3.5-0.8B SFT (Harvest) | ripe 토마토 전체 이미지 + bbox → 최적 수확 대상 선정 |

**시각화 색상**
- ⬜ 흰색 : unripe (YOLO 탐지됐으나 미성숙)
- 🟥 빨간색 : ripe + harvest score 표시
- 🟩 라임색 : 최종 선택 토마토 (SELECTED)

In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5, 6"

import json
import re
import random
import time
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import gradio as gr
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO
from unsloth import FastVisionModel

print("[INFO] Imports complete")

[INFO] Imports complete


In [ ]:
# ── 경로 설정 ──────────────────────────────────────────────────────────────────
_ROOT = Path("/home/seoooa/project/smart-farm-tomato")

YOLO_WEIGHT         = str(_ROOT / "src" / "models" / "yolo_v26_sft_detection" / "weights" / "yolo_v26s_twoclass.pt")
RIPENESS_MODEL_PATH = str(_ROOT / "src" / "models" / "qwen3_5_sft_ripeness" / "weights" / "VLLM_float16" / "qwen3.5_0.8b_lora")
HARVEST_MODEL_PATH  = str(_ROOT / "src" / "models" / "qwen3_5_sft_harvest" / "weights" / "VLLM_float16" / "qwen3.5_0.8b_lora")

HARVEST_TEST_DIR = str(_ROOT / "data" / "Tomato-Harvest-2k-Vision" / "test")
METADATA_PATH    = str(Path(HARVEST_TEST_DIR) / "metadata.jsonl")

YOLO_CONF = 0.7
YOLO_IOU  = 0.45
YOLO_DEVICE = 0   # YOLO GPU 번호 (CUDA_VISIBLE_DEVICES 기준)

print(f"[INFO] YOLO      : {YOLO_WEIGHT}")
print(f"[INFO] Ripeness  : {RIPENESS_MODEL_PATH}")
print(f"[INFO] Harvest   : {HARVEST_MODEL_PATH}")

[INFO] YOLO      : /home/seoooa/project/smart-farm-tomato/src/models/yolo_v26_sft_detection/weights/yolo_v26s_twoclass.pt
[INFO] Ripeness  : /home/seoooa/project/smart-farm-tomato/src/models/qwen3_5_sft_ripeness/weights/VLLM_float16/qwen3.5_0.8b_lora
[INFO] Harvest   : /home/seoooa/project/smart-farm-tomato/src/models/qwen3_5-sft-harvest/weights/VLLM_float16/qwen3.5_0.8b_lora


In [13]:
# ── Ripeness 프롬프트 ──────────────────────────────────────────────────────────
RIPENESS_SYSTEM = """\
You are an expert tomato ripeness classifier.

Definitions:
- ripe: The center tomato shows any visible sign of ripening —
  fully red, predominantly red, or beginning to turn red
  (early blush, orange tint, reddish tint, or partial red coloration).
  It does NOT need to be fully red.

- unripe: The center tomato shows no sign of redness —
  it is fully green, yellow, or clearly pre-ripening.

Rules:
- Focus only on the CENTER tomato. Ignore borders, leaves, stems, and background.
- Base the decision on overall visible color, not tiny local artifacts.
"""

RIPENESS_USER = """\
Classify the CENTER tomato as ripe or unripe.

Return exactly one valid JSON object:
{
  "is_tomato": 1,
  "is_ripe": 0,
  "reasoning": ""
}

Rules:
- "is_tomato": 1 if a tomato is visible, else 0.
- "is_ripe": 1 if ripe, else 0.
- "reasoning": one sentence describing the visual evidence.
- Output only the JSON object, nothing else.
"""

RIPENESS_MAX_NEW_TOKENS = 96
RIPENESS_CROP_SIZE      = 512

# ── Harvest 프롬프트 ───────────────────────────────────────────────────────────
HARVEST_SYSTEM = """\
You are a tomato harvest-selection annotation assistan.

Your task is to evaluate all ripe tomato candidates in the image and generate one output JSON object.
You must assign scores to every tomato listed in ripe_tomatoes and always select the single best tomato for harvest.

For each candidate, assign:
- ripeness_score (0-10): redness and maturity
- visibility_score (0-10): visibility considering leaves, stems, blur, and image border
- isolation_score (0-10): separation from other tomatoes

Rules:
- Compare tomatoes RELATIVE to each other within the same image.
- Set total_score = ripeness_score + visibility_score + isolation_score.
- Always select exactly one candidate ID from ripe_tomatoes.
"""

HARVEST_USER = """\
Choose the best harvest tomato from ripe_tomatoes.

Input:
{{
  "image_size": {image_size},
  "ripe_tomatoes": {ripe_tomatoes}
}}

Example JSON output:
{{
  "selected_tomato_id": 2,
  "reasoning": "Tomato 2 is more uniformly red and clearly isolated than the other ripe candidates."
  "tomato_scores": [
    {{
      "id": 1,
      "ripeness_score": 7,
      "visibility_score": 8,
      "isolation_score": 5,
      "total_score": 20
    }},
    {{
      "id": 2,
      "ripeness_score": 9,
      "visibility_score": 8,
      "isolation_score": 7,
      "total_score": 24
    }}
  ]
}}

Rules:
- Score every tomato in ripe_tomatoes.
- total_score must equal ripeness_score + visibility_score + isolation_score.
- tomato_scores must contain one score object for every tomato in ripe_tomatoes.
- reasoning must be exactly ONE short sentence to explain the selected tomato using comparative evidence based on the scores.
- Output only the JSON object, nothing else.
"""

HARVEST_BASE_TOK     = 100
HARVEST_PER_TOMATO   = 50

print("[INFO] Prompts defined")

[INFO] Prompts defined


In [15]:
# ── YOLO ──────────────────────────────────────────────────────────────────────
if "yolo_model" not in dir() or yolo_model is None:
    yolo_model = YOLO(YOLO_WEIGHT)
    print(f"[INFO] YOLO loaded: {YOLO_WEIGHT}")

# ── Ripeness ──────────────────────────────────────────────────────────────────
if "ripeness_model" not in dir() or ripeness_model is None:
    ripeness_model, ripeness_tokenizer = FastVisionModel.from_pretrained(
        RIPENESS_MODEL_PATH, load_in_4bit=False
    )
    FastVisionModel.for_inference(ripeness_model)
    print(f"[INFO] Ripeness model loaded: {RIPENESS_MODEL_PATH}")

# ── Harvest ───────────────────────────────────────────────────────────────────
if "harvest_model" not in dir() or harvest_model is None:
    harvest_model, harvest_tokenizer = FastVisionModel.from_pretrained(
        HARVEST_MODEL_PATH, load_in_4bit=False
    )
    FastVisionModel.for_inference(harvest_model)
    print(f"[INFO] Harvest model loaded: {HARVEST_MODEL_PATH}")

RuntimeError: Unsloth: No config file found - are you sure the `model_name` is correct?
If you're using a model on your local device, confirm if the folder location exists.
If you're using a HuggingFace online model, check if it exists.

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  Step 1 — YOLO Detection
# ════════════════════════════════════════════════════════════════════

@dataclass
class Detection:
    confidence: float
    xyxy: list[float] = field(default_factory=list)  # [x1, y1, x2, y2]

def step1_detect(pil_image: Image.Image) -> list[Detection]:
    """PIL 이미지에서 토마토를 탐지하고 단일 클래스로 통합합니다."""
    results = yolo_model.predict(
        source=pil_image,
        conf=YOLO_CONF,
        iou=YOLO_IOU,
        device=YOLO_DEVICE,
        verbose=False,
        save=False,
    )
    detections = []
    r0 = results[0]
    if r0.boxes is None or len(r0.boxes) == 0:
        return detections
    for xyxy, conf in zip(r0.boxes.xyxy.cpu().numpy(), r0.boxes.conf.cpu().numpy()):
        detections.append(Detection(confidence=float(conf), xyxy=[float(v) for v in xyxy]))
    return detections


def _crop_bbox(
    pil_image: Image.Image,
    xyxy: list[float],
    pad: int = 4,
) -> tuple[Image.Image, tuple[int, int, int, int]]:
    """PIL 이미지에서 bbox crop (패딩 포함)."""
    W, H = pil_image.size
    x1 = max(0, int(xyxy[0]) - pad)
    y1 = max(0, int(xyxy[1]) - pad)
    x2 = min(W, int(xyxy[2]) + pad)
    y2 = min(H, int(xyxy[3]) + pad)
    return pil_image.crop((x1, y1, x2, y2)), (x1, y1, x2, y2)


# ════════════════════════════════════════════════════════════════════
#  Step 2 — Ripeness Classification
# ════════════════════════════════════════════════════════════════════

@dataclass
class RipenessResult:
    is_tomato: int
    is_ripe:   int
    reasoning: str
    raw: str = ""

    @property
    def label(self) -> str:
        if not self.is_tomato:
            return "not_tomato"
        return "ripe" if self.is_ripe else "unripe"

def _parse_json(raw: str):
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            return json.loads(m.group())
    except Exception:
        pass
    return None

def step2_ripeness(
    pil_image: Image.Image,
    detections: list[Detection],
    batch_size: int = 8,
) -> list[tuple[tuple[int,int,int,int], float, RipenessResult | None]]:
    """탐지된 모든 bbox를 crop한 뒤 배치로 익음도를 분류합니다.

    Returns: [(bbox, det_conf, RipenessResult | None), ...]
    """
    crops, bboxes, confs = [], [], []
    for det in detections:
        crop, bbox = _crop_bbox(pil_image, det.xyxy)
        crops.append(crop.resize((RIPENESS_CROP_SIZE, RIPENESS_CROP_SIZE)))
        bboxes.append(bbox)
        confs.append(det.confidence)

    ripeness_tokenizer.padding_side = "left"
    msgs_template = [
        {"role": "system", "content": [{"type": "text", "text": RIPENESS_SYSTEM}]},
        {"role": "user",   "content": [{"type": "image"}, {"type": "text", "text": RIPENESS_USER}]},
    ]

    all_results: list[RipenessResult | None] = []
    for start in range(0, len(crops), batch_size):
        batch_imgs = crops[start : start + batch_size]
        texts = [
            ripeness_tokenizer.apply_chat_template(
                msgs_template, add_generation_prompt=True, enable_thinking=False
            )
            for _ in batch_imgs
        ]
        inputs = ripeness_tokenizer(
            batch_imgs, texts,
            padding=True, add_special_tokens=False, return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            out_ids = ripeness_model.generate(
                **inputs, max_new_tokens=RIPENESS_MAX_NEW_TOKENS, do_sample=False, use_cache=True
            )

        input_len = inputs["input_ids"].shape[1]
        for out in out_ids:
            raw = ripeness_tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
            parsed = _parse_json(raw)
            all_results.append(
                RipenessResult(
                    is_tomato=int(parsed.get("is_tomato", 1)),
                    is_ripe=int(parsed.get("is_ripe", 0)),
                    reasoning=parsed.get("reasoning", ""),
                    raw=raw,
                ) if parsed else None
            )

    return list(zip(bboxes, confs, all_results))


# ════════════════════════════════════════════════════════════════════
#  Step 3 — Harvest Selection
# ════════════════════════════════════════════════════════════════════

@dataclass
class TomatoScore:
    id: int
    ripeness_score:   int
    visibility_score: int
    isolation_score:  int
    total_score:      int

@dataclass
class HarvestResult:
    selected_tomato_id: int
    reasoning: str
    tomato_scores: list[TomatoScore] = field(default_factory=list)
    raw: str = ""

def step3_harvest(
    pil_image: Image.Image,
    ripeness_results: list[tuple],
) -> HarvestResult | None:
    """ripe 토마토만 골라 전체 이미지에서 최적 수확 대상을 선정합니다."""
    ripe_tomatoes = [
        {"id": i + 1, "bbox": list(bbox)}
        for i, (bbox, _conf, result) in enumerate(ripeness_results)
        if result is not None and result.label == "ripe"
    ]
    if not ripe_tomatoes:
        return None

    prompt_text = HARVEST_USER.format(
        image_size=json.dumps(list(pil_image.size)),
        ripe_tomatoes=json.dumps(ripe_tomatoes, ensure_ascii=False),
    )
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": HARVEST_SYSTEM}]},
        {"role": "user",   "content": [{"type": "image"}, {"type": "text", "text": prompt_text}]},
    ]
    input_text = harvest_tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, enable_thinking=False
    )
    inputs = harvest_tokenizer(
        pil_image, input_text, add_special_tokens=False, return_tensors="pt"
    ).to("cuda")

    max_new_tokens = HARVEST_BASE_TOK + HARVEST_PER_TOMATO * len(ripe_tomatoes)
    with torch.no_grad():
        out_ids = harvest_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True
        )
    raw = harvest_tokenizer.decode(
        out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    parsed = _parse_json(raw)
    if parsed is None:
        return None

    scores = [
        TomatoScore(
            id=s.get("id", -1),
            ripeness_score=s.get("ripeness_score", 0),
            visibility_score=s.get("visibility_score", 0),
            isolation_score=s.get("isolation_score", 0),
            total_score=s.get("total_score", 0),
        )
        for s in parsed.get("tomato_scores", []) if isinstance(s, dict)
    ]
    return HarvestResult(
        selected_tomato_id=parsed.get("selected_tomato_id", -1),
        reasoning=parsed.get("reasoning", ""),
        tomato_scores=scores,
        raw=raw,
    )

print("[INFO] Pipeline step functions defined")

[INFO] Pipeline step functions defined


In [ ]:
# 3색 bbox 시각화 (PIL 기반)
# ⬜ 흰색 : unripe   🟥 빨간색 : ripe + score   🟩 라임 : SELECTED

_COLOR_UNRIPE   = "white"
_COLOR_RIPE     = "red"
_COLOR_SELECTED = "lime"
_FONT_PATH      = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
_FONT_SIZE      = 18


def _get_font(size: int = _FONT_SIZE):
    try:
        return ImageFont.truetype(_FONT_PATH, size)
    except Exception:
        return ImageFont.load_default()


def _draw_label(draw: ImageDraw.ImageDraw, text: str, x: int, y: int, font):
    """검정 배경 + 노란 텍스트 라벨을 그립니다."""
    bb   = draw.textbbox((0, 0), text, font=font)
    tw   = bb[2] - bb[0]
    th   = bb[3] - bb[1]
    tx   = x + 2
    ty   = max(0, y - th - 4)
    draw.rectangle([tx - 2, ty - 2, tx + tw + 2, ty + th + 2], fill="black")
    draw.text((tx, ty), text, fill="yellow", font=font)


def draw_pipeline_image(
    pil_image: Image.Image,
    ripeness_results: list[tuple],
    harvest_result: HarvestResult | None,
) -> Image.Image:
    """파이프라인 결과를 3색 bbox로 시각화한 PIL 이미지를 반환합니다."""
    vis  = pil_image.copy().convert("RGB")
    draw = ImageDraw.Draw(vis)
    font = _get_font()

    selected_id = harvest_result.selected_tomato_id if harvest_result else -1
    score_map   = (
        {s.id: s.total_score for s in harvest_result.tomato_scores}
        if harvest_result else {}
    )

    for i, (bbox, _conf, result) in enumerate(ripeness_results):
        tid   = i + 1
        label = result.label if result else "unknown"
        x1, y1, x2, y2 = bbox

        if tid == selected_id:
            color = _COLOR_SELECTED
            width = 3
            score = score_map.get(tid)
            tag   = f"id:{tid} SELECTED" + (f" score:{score}" if score is not None else "")
        elif label == "ripe":
            color = _COLOR_RIPE
            width = 2
            score = score_map.get(tid)
            tag   = f"id:{tid} ripe" + (f" score:{score}" if score is not None else "")
        else:
            color = _COLOR_UNRIPE
            width = 2
            tag   = f"id:{tid} {label}"

        draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
        _draw_label(draw, tag, x1, y1, font)

    return vis

print("[INFO] Visualization function defined")

[INFO] Visualization function defined


In [ ]:
def run_pipeline(
    pil_image: Image.Image,
) -> tuple[Image.Image, str, str, str]:
    """이미지 1장에 대해 전체 파이프라인을 실행합니다.

    Returns:
        (annotated_image, ripeness_json_str, harvest_json_str, timing_str)
    """
    timings: dict[str, float] = {}

    # Step 1: YOLO detection
    t0 = time.perf_counter()
    detections = step1_detect(pil_image)
    timings["1. YOLO detection"] = time.perf_counter() - t0

    if not detections:
        empty    = json.dumps({"message": "탐지된 토마토 없음"}, ensure_ascii=False, indent=2)
        timing_str = _format_timings(timings, n_detected=0, n_ripe=0)
        return pil_image, empty, empty, timing_str

    # Step 2: Ripeness classification
    t0 = time.perf_counter()
    ripeness_results = step2_ripeness(pil_image, detections)
    timings["2. Qwen Ripeness"] = time.perf_counter() - t0

    n_ripe = sum(1 for _, _, r in ripeness_results if r and r.label == "ripe")

    # Step 3: Harvest selection
    t0 = time.perf_counter()
    harvest_result = step3_harvest(pil_image, ripeness_results)
    timings["3. Qwen Harvest"] = time.perf_counter() - t0

    # Visualization
    annotated = draw_pipeline_image(pil_image, ripeness_results, harvest_result)

    # JSON 출력 포맷
    ripeness_out = [
        {
            "id":        i + 1,
            "bbox":      list(bbox),
            "det_conf":  round(conf, 4),
            "label":     result.label if result else "parse_error",
            "is_ripe":   result.is_ripe if result else None,
            "reasoning": result.reasoning if result else None,
        }
        for i, (bbox, conf, result) in enumerate(ripeness_results)
    ]

    id_to_bbox = {e["id"]: e["bbox"] for e in ripeness_out}
    harvest_out = None
    if harvest_result:
        harvest_out = {
            "selected_tomato_id": harvest_result.selected_tomato_id,
            "selected_bbox":      id_to_bbox.get(harvest_result.selected_tomato_id),
            "reasoning":          harvest_result.reasoning,
            "tomato_scores": [
                {
                    "id":               s.id,
                    "ripeness_score":   s.ripeness_score,
                    "visibility_score": s.visibility_score,
                    "isolation_score":  s.isolation_score,
                    "total_score":      s.total_score,
                }
                for s in harvest_result.tomato_scores
            ],
        }

    timing_str = _format_timings(
        timings,
        n_detected=len(detections),
        n_ripe=n_ripe,
    )

    return (
        annotated,
        json.dumps(ripeness_out, ensure_ascii=False, indent=2),
        json.dumps(harvest_out,  ensure_ascii=False, indent=2),
        timing_str,
    )


def _format_timings(timings: dict[str, float], n_detected: int, n_ripe: int) -> str:
    total = sum(timings.values())
    lines = ["⏱  Step-by-Step Timing\n" + "─" * 36]
    for name, sec in timings.items():
        lines.append(f"  {name:<22}  {sec:6.2f} s")
    lines.append("─" * 36)
    lines.append(f"  {'Total':<22}  {total:6.2f} s")
    lines.append("")
    lines.append(f"  Detected : {n_detected} tomatoes")
    lines.append(f"  Ripe     : {n_ripe} tomatoes")
    return "\n".join(lines)


print("[INFO] run_pipeline defined")

[INFO] run_pipeline defined


In [ ]:
# ── Test set에서 랜덤 10개 샘플 로드 ──────────────────────────────────────────
random.seed(42)
with open(METADATA_PATH) as f:
    _all_samples = [json.loads(line) for line in f if line.strip()]
_sampled = random.sample(_all_samples, min(10, len(_all_samples)))

_sample_names = [
    f"Sample {i+1}:  {s['file_name'][:55]}"
    for i, s in enumerate(_sampled)
]

def _load_sample(name: str | None):
    if name is None:
        return None
    idx = _sample_names.index(name)
    return Image.open(str(Path(HARVEST_TEST_DIR) / _sampled[idx]["file_name"]))

print(f"[INFO] {len(_sampled)} test examples loaded")


# ── Gradio UI ─────────────────────────────────────────────────────────────────
def predict(pil_image):
    if pil_image is None:
        empty = "이미지를 입력하세요."
        return None, empty, empty, ""
    annotated, rip_json, hvs_json, timing_str = run_pipeline(pil_image)
    return annotated, rip_json, hvs_json, timing_str


with gr.Blocks(title="🍅 Tomato Pipeline Demo") as demo:
    gr.Markdown(
        "# 🍅 Tomato Pipeline Demo\n"
        "**YOLO Detection → Qwen3.5 Ripeness → Qwen3.5 Harvest Selection**\n\n"
        "드롭다운에서 샘플을 선택하거나 직접 이미지를 업로드하세요."
    )

    # ── 입력 영역 ──────────────────────────────────────────────────────────────
    sample_dd = gr.Dropdown(
        choices=_sample_names,
        label="Test Set Examples (랜덤 10개) — 선택하면 자동 로드",
        value=None,
    )

    with gr.Row():
        img_input = gr.Image(type="pil", label="입력 이미지 (업로드 또는 샘플 선택)")

    sample_dd.change(
        fn=_load_sample,
        inputs=sample_dd,
        outputs=img_input,
    )

    run_btn = gr.Button("▶  Run Pipeline", variant="primary", size="lg")

    # ── 출력 영역 ──────────────────────────────────────────────────────────────
    with gr.Row():
        img_output  = gr.Image(type="pil", label="Pipeline 결과 시각화")
        out_timing  = gr.Textbox(lines=12, label="⏱  Timing", interactive=False)

    with gr.Row():
        out_ripeness = gr.Textbox(lines=20, label="Step 2 · Ripeness 결과 (JSON)")
        out_harvest  = gr.Textbox(lines=20, label="Step 3 · Harvest 결과 (JSON)")

    run_btn.click(
        fn=predict,
        inputs=img_input,
        outputs=[img_output, out_ripeness, out_harvest, out_timing],
    )

demo.launch(share=True)

[INFO] 10 test examples loaded


* Running on local URL:  http://127.0.0.1:7871
* Running on public URL: https://cb049492bb0e113cfb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
